In [6]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
from data.combine_timetable import combine_timetables
from data.timetable import load_gold, save_gold


In [7]:
n_freight_trains = 300
combine_timetables(n_freight_trains)

,SECTION,SOURCE,TARGET,PLANNED_DEPARTURE,PLANNED_ARRIVAL,TRAIN_NO,TYPE,DYNAMICS,PERIOD,TRAIN_TYPE,ENTRY_SECONDS,EXIT_SECONDS
0,36N:SCHAARBEEK-BRUSSEL-NOORD,SCHAARBEEK,BRUSSEL-NOORD,2025-01-01 21:23:00,2025-01-01 21:26:00,10,SOURCE,ACC-BR,EVENING,ICE,76980,77160
1,SCHAARBEEK -- ICE: FRANKFURT(MAIN) HBF -> BRUS...,SCHAARBEEK,SCHAARBEEK,2025-01-01 21:28:00,2025-01-01 21:26:00,10,WITHIN-STATION-DWELL,ACC-0,EVENING,ICE,77160,77280
2,0-2:BRUSSEL-NOORD-BRUSSEL-CONGRES,BRUSSEL-NOORD,BRUSSEL-CONGRES,2025-01-01 21:28:00,2025-01-01 21:30:00,10,BETWEEN-STATION,ACC-0,EVENING,ICE,77280,77400
3,BRUSSEL-NOORD -- platform 5,BRUSSEL-NOORD,BRUSSEL-NOORD,2025-01-01 21:30:00,2025-01-01 21:30:00,10,WITHIN-STATION-PASSING,0-0,EVENING,ICE,77400,77405
4,0-2:BRUSSEL-CONGRES-BRUSSEL-CENTRAAL,BRUSSEL-CONGRES,BRUSSEL-CENTRAAL,2025-01-01 21:30:00,2025-01-01 21:31:00,10,BETWEEN-STATION,0-0,EVENING,ICE,77400,77460
...,...,...,...,...,...,...,...,...,...,...,...,...
11975,THURN EN TAXIS,THURN EN TAXIS,THURN EN TAXIS,2025-01-01 20:56:47,2025-01-01 20:55:47,900298,WITHIN-STATION-DWELL,0-0,EVENING,freight,75347,75407
11976,28:THURN EN TAXIS-SCHAARBEEK,THURN EN TAXIS,SCHAARBEEK,2025-01-01 20:56:47,2025-01-01 21:03:40,900298,BETWEEN-STATION,0-0,EVENING,freight,75407,75820
11977,SCHAARBEEK,SCHAARBEEK,SCHAARBEEK,2025-01-01 21:04:40,2025-01-01 21:03:40,900298,WITHIN-STATION-DWELL,0-0,EVENING,freight,75820,75880
11978,36N:SCHAARBEEK-BRUSSEL-NOORD,SCHAARBEEK,BRUSSEL-NOORD,2025-01-01 21:04:40,2025-01-01 21:08:34,900298,BETWEEN-STATION,0-0,EVENING,freight,75880,76114


In [8]:
passenger = load_gold('passenger')
freight   = load_gold('freight', n_trains=n_freight_trains)

# Validatie
overlap = set(passenger['TRAIN_NO']) & set(freight['TRAIN_NO'])
print(f"Overlappende TRAIN_NO: {overlap if overlap else 'geen'}")

combined = combine_timetables(n_freight_trains)
print(combined['TRAIN_TYPE'].value_counts())
print(f"ENTRY_SECONDS range: {combined['ENTRY_SECONDS'].min():.0f}s — {combined['ENTRY_SECONDS'].max():.0f}s")

Overlappende TRAIN_NO: geen
TRAIN_TYPE
IC         6669
L          3514
freight    1116
EURST       369
INT         179
ICE         133
Name: count, dtype: int64
ENTRY_SECONDS range: 0s — 87258s


In [9]:
before   = len(combined)
combined = combined[combined['EXIT_SECONDS'] > combined['ENTRY_SECONDS']]
removed  = before - len(combined)
print(f"Verwijderd: {removed} segmenten met EXIT_SECONDS <= ENTRY_SECONDS")

Verwijderd: 3 segmenten met EXIT_SECONDS <= ENTRY_SECONDS


In [10]:
save_gold(combined, source='combined', n_trains=n_freight_trains)

Opgeslagen (combined): 1319 treinen, 11977 segmenten → /Users/ddw/Desktop/Rescheduling/data/gold/combined/300
